# Part A: Data Understanding



In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report
from sklearn.feature_selection import mutual_info_classif, chi2, f_classif
from sklearn.preprocessing import LabelEncoder
from scipy.stats import zscore

df = pd.read_csv(r"C:\Users\varsh\OneDrive\Desktop\ds\data\hotel_bookings.csv")
print("Business Problem:")
print("The goal is to predict whether a hotel booking will be canceled or not.")
print("Hotels can use this prediction to reduce revenue loss and manage room availability better.")

print("\nDataset Shape:")
print(df.shape)

print("\nTarget Variable:")
target = "is_canceled"
print(target)

print("\nProblem Type:")
print("This is a classification problem because the target has classes: canceled or not canceled.")

print("\nFirst 5 Rows:")
display(df.head())

print("\nColumn Names:")
print(df.columns.tolist())

print("\nData Types:")
print(df.dtypes)

date_features = [col for col in df.columns if "date" in col.lower()]
binary_features = [col for col in df.columns if df[col].nunique() == 2]
numerical_features = df.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = df.select_dtypes(include=["object"]).columns.tolist()

print("\nDate Features:")
print(date_features)

print("\nBinary Features:")
print(binary_features)

print("\nNumerical Features:")
print(numerical_features)

print("\nCategorical Features:")
print(categorical_features)

feature_explanation = {
    "hotel": "Type of hotel",
    "is_canceled": "Booking cancellation status",
    "lead_time": "Days between booking date and arrival date",
    "arrival_date_year": "Arrival year",
    "arrival_date_month": "Arrival month",
    "arrival_date_week_number": "Arrival week number",
    "arrival_date_day_of_month": "Arrival day of month",
    "stays_in_weekend_nights": "Weekend nights stayed",
    "stays_in_week_nights": "Week nights stayed",
    "adults": "Number of adults",
    "children": "Number of children",
    "babies": "Number of babies",
    "meal": "Meal type booked",
    "country": "Guest country",
    "market_segment": "Market segment",
    "distribution_channel": "Booking distribution channel",
    "is_repeated_guest": "Whether guest is repeated",
    "previous_cancellations": "Previous cancellations by guest",
    "previous_bookings_not_canceled": "Previous non-canceled bookings",
    "reserved_room_type": "Reserved room type",
    "assigned_room_type": "Assigned room type",
    "booking_changes": "Number of booking changes",
    "deposit_type": "Deposit type",
    "agent": "Agent ID",
    "company": "Company ID",
    "days_in_waiting_list": "Days in waiting list",
    "customer_type": "Customer type",
    "adr": "Average daily rate",
    "required_car_parking_spaces": "Required parking spaces",
    "total_of_special_requests": "Total special requests",
    "reservation_status": "Final reservation status",
    "reservation_status_date": "Date of reservation status"
}

print("\nFeature Explanation:")
for col in df.columns:
    print(col, ":", feature_explanation.get(col, "Feature from dataset"))

# Part B: Exploratory Data Analysis




In [ ]:
df_eda = df.copy()

print("Dataset Overview")
print("Rows and Columns:", df_eda.shape)
print("\nDuplicate Rows:", df_eda.duplicated().sum())

print("\nMissing Values Count:")
print(df_eda.isnull().sum().sort_values(ascending=False))

print("\nMissing Values Percentage:")
missing_percent = (df_eda.isnull().mean() * 100).sort_values(ascending=False)
print(missing_percent[missing_percent > 0])

df_eda["children"] = df_eda["children"].fillna(0)
df_eda["country"] = df_eda["country"].fillna("Unknown")
df_eda["agent"] = df_eda["agent"].fillna(0)
df_eda["company"] = df_eda["company"].fillna(0)

print("\nMissing Values After Treatment:")
print(df_eda.isnull().sum().sum())

df_eda["total_guests"] = df_eda["adults"] + df_eda["children"] + df_eda["babies"]
df_eda["total_nights"] = df_eda["stays_in_weekend_nights"] + df_eda["stays_in_week_nights"]
df_eda["is_family"] = np.where((df_eda["children"] > 0) | (df_eda["babies"] > 0), 1, 0)

print("\nEngineered Features Created:")
print(["total_guests", "total_nights", "is_family"])

num_cols = ["lead_time", "adr", "total_guests", "total_nights", "booking_changes", "days_in_waiting_list", "total_of_special_requests"]
num_cols = [col for col in num_cols if col in df_eda.columns]

for col in num_cols:
    plt.figure(figsize=(6, 4))
    df_eda[col].hist(bins=30)
    plt.title("Histogram of " + col)
    plt.xlabel(col)
    plt.ylabel("Count")
    plt.show()

for col in num_cols:
    plt.figure(figsize=(6, 4))
    df_eda[col].plot(kind="kde")
    plt.title("KDE Plot of " + col)
    plt.xlabel(col)
    plt.show()

for col in num_cols:
    plt.figure(figsize=(6, 4))
    df_eda.boxplot(column=col)
    plt.title("Boxplot of " + col)
    plt.ylabel(col)
    plt.show()

cat_cols = ["hotel", "arrival_date_month", "meal", "market_segment", "distribution_channel", "deposit_type", "customer_type", "reserved_room_type"]
cat_cols = [col for col in cat_cols if col in df_eda.columns]

for col in cat_cols:
    plt.figure(figsize=(8, 4))
    df_eda[col].value_counts().head(10).plot(kind="bar")
    plt.title("Count Plot of " + col)
    plt.xlabel(col)
    plt.ylabel("Count")
    plt.xticks(rotation=45)
    plt.show()

print("\nIQR Outlier Detection:")
for col in num_cols:
    q1 = df_eda[col].quantile(0.25)
    q3 = df_eda[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    outliers = df_eda[(df_eda[col] < lower) | (df_eda[col] > upper)].shape[0]
    print(col, ":", outliers)

print("\nZ-score Outlier Detection:")
for col in num_cols:
    values = df_eda[col].replace([np.inf, -np.inf], np.nan).fillna(df_eda[col].median())
    z_scores = np.abs(zscore(values))
    print(col, ":", (z_scores > 3).sum())

print("\nTarget Variable Analysis:")
print(df_eda["is_canceled"].value_counts())
print(df_eda["is_canceled"].value_counts(normalize=True) * 100)

plt.figure(figsize=(5, 4))
df_eda["is_canceled"].value_counts().plot(kind="bar")
plt.title("Target Variable Count")
plt.xlabel("is_canceled")
plt.ylabel("Count")
plt.show()

corr_cols = num_cols + ["is_canceled"]
corr = df_eda[corr_cols].corr()

plt.figure(figsize=(8, 6))
plt.imshow(corr)
plt.colorbar()
plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.columns)), corr.columns)
plt.title("Correlation Heatmap")
plt.show()

plt.figure(figsize=(6, 4))
plt.scatter(df_eda["lead_time"], df_eda["adr"], alpha=0.2)
plt.title("Scatter Plot: Lead Time vs ADR")
plt.xlabel("lead_time")
plt.ylabel("adr")
plt.show()

print("\nCancellation Rate by Categorical Features:")
for col in cat_cols:
    print("\n", col)
    print(df_eda.groupby(col)["is_canceled"].mean().sort_values(ascending=False).head(10))

feature_df = df_eda.copy()
drop_cols = ["reservation_status", "reservation_status_date"]
feature_df = feature_df.drop(columns=drop_cols, errors="ignore")

for col in feature_df.select_dtypes(include=["object"]).columns:
    feature_df[col] = LabelEncoder().fit_transform(feature_df[col].astype(str))

X_temp = feature_df.drop("is_canceled", axis=1)
y_temp = feature_df["is_canceled"]

mi_scores = mutual_info_classif(X_temp.fillna(0), y_temp, random_state=42)
mi_result = pd.DataFrame({"Feature": X_temp.columns, "Mutual Information": mi_scores})
mi_result = mi_result.sort_values(by="Mutual Information", ascending=False)

print("\nTop Features using Mutual Information:")
display(mi_result.head(10))

# Part C: Data Preprocessing


In [ ]:
df_model = df.copy()

df_model["children"] = df_model["children"].fillna(0)
df_model["country"] = df_model["country"].fillna("Unknown")
df_model["agent"] = df_model["agent"].fillna(0)
df_model["company"] = df_model["company"].fillna(0)

df_model["total_guests"] = df_model["adults"] + df_model["children"] + df_model["babies"]
df_model["total_nights"] = df_model["stays_in_weekend_nights"] + df_model["stays_in_week_nights"]
df_model["is_family"] = np.where((df_model["children"] > 0) | (df_model["babies"] > 0), 1, 0)

df_model = df_model[df_model["total_guests"] > 0]
df_model = df_model[df_model["adr"] >= 0]

leakage_cols = ["reservation_status", "reservation_status_date"]
df_model = df_model.drop(columns=leakage_cols, errors="ignore")

outlier_cols = ["lead_time", "adr", "total_nights", "days_in_waiting_list"]

for col in outlier_cols:
    q1 = df_model[col].quantile(0.25)
    q3 = df_model[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    df_model[col] = df_model[col].clip(lower, upper)

X = df_model.drop("is_canceled", axis=1)
y = df_model["is_canceled"]

X = pd.get_dummies(X, drop_first=True)

print("Feature Scaling:")
print("Decision Trees do not require feature scaling because they split data based on feature values.")

print("\nClass Balance:")
print(y.value_counts(normalize=True) * 100)
print("Class imbalance will be handled using class_weight='balanced' in the Decision Tree model.")

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.40, random_state=42, stratify=y
)

X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print("\nTraining Set:", X_train.shape)
print("Validation Set:", X_valid.shape)
print("Test Set:", X_test.shape)

print("\nPreprocessing Completed")

# Part D: Decision Tree Modeling



In [ ]:
def evaluate_model(model, X_data, y_data):
    y_pred = model.predict(X_data)
    y_prob = model.predict_proba(X_data)[:, 1]

    result = {
        "Accuracy": accuracy_score(y_data, y_pred),
        "Precision": precision_score(y_data, y_pred),
        "Recall": recall_score(y_data, y_pred),
        "F1-score": f1_score(y_data, y_pred),
        "ROC-AUC": roc_auc_score(y_data, y_prob)
    }

    return result, y_pred

baseline_tree = DecisionTreeClassifier(random_state=42, class_weight="balanced")
baseline_tree.fit(X_train, y_train)

baseline_result, baseline_pred = evaluate_model(baseline_tree, X_valid, y_valid)

print("Baseline Decision Tree Results:")
display(pd.DataFrame([baseline_result]))

print("\nBaseline Confusion Matrix:")
print(confusion_matrix(y_valid, baseline_pred))

print("\nBaseline Classification Report:")
print(classification_report(y_valid, baseline_pred))

param_grid = {
    "criterion": ["gini", "entropy"],
    "max_depth": [3, 5, 7, 10, 15],
    "min_samples_split": [2, 10, 20],
    "min_samples_leaf": [1, 5, 10, 20]
}

grid_search = GridSearchCV(
    DecisionTreeClassifier(random_state=42, class_weight="balanced"),
    param_grid,
    cv=3,
    scoring="f1",
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

best_tree = grid_search.best_estimator_

print("\nBest Parameters:")
print(grid_search.best_params_)

tuned_result, tuned_pred = evaluate_model(best_tree, X_valid, y_valid)

print("\nTuned Decision Tree Results:")
display(pd.DataFrame([tuned_result]))

print("\nTuned Confusion Matrix:")
print(confusion_matrix(y_valid, tuned_pred))

print("\nTuned Classification Report:")
print(classification_report(y_valid, tuned_pred))

comparison = pd.DataFrame([baseline_result, tuned_result], index=["Baseline", "Tuned"])
print("\nBaseline vs Tuned Model:")
display(comparison)

# Part E: Model Interpretation




In [ ]:
plt.figure(figsize=(20, 10))
plot_tree(
    best_tree,
    feature_names=X_train.columns,
    class_names=["Not Canceled", "Canceled"],
    filled=True,
    max_depth=3,
    fontsize=8
)
plt.title("Decision Tree Visualization")
plt.show()

importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": best_tree.feature_importances_
})

importance = importance.sort_values(by="Importance", ascending=False)

print("Top 15 Important Features:")
display(importance.head(15))

plt.figure(figsize=(8, 6))
importance.head(15).sort_values(by="Importance").plot(
    x="Feature",
    y="Importance",
    kind="barh",
    legend=False
)
plt.title("Top 15 Feature Importances")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.show()

rules = export_text(best_tree, feature_names=list(X_train.columns), max_depth=3)

print("Decision Rules from the Tree:")
print(rules)

print("\nFive Rule Interpretations:")
print("1. Lead time is important because bookings made far in advance may have higher cancellation risk.")
print("2. Deposit type is important because customers with no deposit may cancel more easily.")
print("3. Special requests are important because more requests may indicate stronger booking intention.")
print("4. Total nights can affect cancellation because longer stays may have different cancellation behavior.")
print("5. Market segment is important because online, offline, and group bookings may cancel differently.")

# Part F: Model Evaluation



In [ ]:
test_result, test_pred = evaluate_model(best_tree, X_test, y_test)

final_results = pd.DataFrame(
    [baseline_result, tuned_result, test_result],
    index=["Baseline Validation", "Tuned Validation", "Tuned Test"]
)

print("Final Model Evaluation:")
display(final_results)

print("\nFinal Test Confusion Matrix:")
print(confusion_matrix(y_test, test_pred))

print("\nFinal Test Classification Report:")
print(classification_report(y_test, test_pred))

print("\nMost Appropriate Metric:")
print("F1-score is very useful because cancellation prediction needs a balance between precision and recall.")
print("Recall is also important because hotels want to correctly identify as many canceled bookings as possible.")
print("ROC-AUC is useful to measure how well the model separates canceled and not canceled bookings.")